# Generation of LCA indicators and associated .mod and .dat files

In [1]:
# %pip install brightway2
# %pip install mescal
# %pip install pypardiso
# %pip install energyscope

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

In [4]:
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '00_Shared'))

In [5]:
import os
import pandas as pd
import bw2data as bd
from mescal import *
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from utils import (
    COMMON_DATA_DIR,
    add_biogenic_climate_change_to_impact_scores_df,
    add_rhhd_and_reqd_to_impact_scores_df,
    compute_territorial_emissions,
    update_ampl_files,
)
from shared.utils import run_model, load_snapshot

In [6]:
ei_version = '3.12'
year = 2050
ssp_rcp = 'SSP1-L' if year == 2050 else None
iam = 'image' if year == 2050 else None

The assessment type `base_wo_iam` and the column 'IAM assumptions' only applies when `year` is set to 2050.

In [7]:
LCA_RESULTS_DIR = f'../03_Results/LCA/{year}/{iam}/{ssp_rcp}/' if year == 2050 else f'../03_Results/LCA/{year}/'
AMPL_FILES_DATA_DIR = f'../02_AMPL_files/data/{year}/{iam}/{ssp_rcp}/' if year == 2050 else f'../02_AMPL_files/data/{year}/'
AMPL_FILES_MODEL_DIR = f'../02_AMPL_files/model/'

## Initialize the EnergyScope model

In [8]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [9]:
# Initialize the 2023 QC model with .mod and .dat files
model = load_snapshot(year)

In [10]:
# Solve the model and get results
results = run_model(model)

Gurobi 12.0.0: 

## Load data and initialize the ESM class

In [11]:
# Load the data
mapping = pd.read_csv('./Data/mapping.csv')
unit_conversion = pd.read_excel('./Data/unit_conversion.xlsx')  # open file and press enter in one computation cell to avoid misreading
techno_compositions = pd.read_csv(COMMON_DATA_DIR / 'technology_compositions.csv')
tech_specifics = pd.read_csv('./Data/technology_specifics.csv')
efficiency = pd.read_csv(COMMON_DATA_DIR / 'efficiency.csv')
lifetime = pd.read_csv(COMMON_DATA_DIR / 'lifetime.csv')
mapping_es_flows_to_cpc = pd.read_csv(COMMON_DATA_DIR / 'mapping_esm_flows_to_CPC.csv')
impact_abbrev = pd.read_csv(COMMON_DATA_DIR / 'impact_abbrev.csv')
mapping_new_product_to_cpc = pd.read_csv(COMMON_DATA_DIR / 'mapping_new_products_to_CPC.csv')

In [12]:
# Load the model from energyscope model
model = results.parameters['layers_in_out'].reset_index().rename(columns={'index0': 'Name', 'index1': 'Flow', 'layers_in_out': 'Amount'}).drop(columns=['Run'])
model = model[model['Amount'] != 0]
model.to_csv(COMMON_DATA_DIR / f'model_{year}.csv', index=False)

In [13]:
# Set up your Brightway project
bd.projects.set_current(f'ecoinvent{ei_version}')

In [14]:
# Database names
name_main_database = f'ecoinvent_cutoff_{ei_version}_{iam}_{ssp_rcp}_{year} - regionalized' if year == 2050 else f'ecoinvent_cutoff_{ei_version}_{year} - regionalized'
name_regiopremise_database = f"regiopremise_{ei_version}_{iam}_{ssp_rcp}_{year}" if year == 2050 else f"regiopremise_{ei_version}_{year}"
name_biosphere_db = 'biosphere3'
name_spatialized_biosphere_db = 'biosphere3_spatialized_flows'
name_es_database = f'EnergyScope_CA-QC_{iam}_{ssp_rcp}_{year}' if year == 2050 else f'EnergyScope_CA-QC_{year}'

In [15]:
regionalize_foregrounds = ['Operation', 'Resource']

In [16]:
main_db = Database(db_names=[name_main_database, name_regiopremise_database], create_pickle=True)
spatialized_biosphere_db = Database(db_names=name_spatialized_biosphere_db)

2026-09-08 16:22:41,977 - Database - INFO - Loaded ecoinvent_cutoff_3.12_image_SSP1-L_2050 - regionalized from pickle!
2026-09-08 16:22:42,488 - Database - INFO - Loaded regiopremise_3.12_image_SSP1-L_2050 from pickle!


Getting activity data


100%|██████████| 111635/111635 [00:01<00:00, 93789.80it/s] 


Adding exchange data to activities


0it [00:00, ?it/s]


Filling out exchange data


100%|██████████| 111635/111635 [00:00<00:00, 4054686.37it/s]
2026-09-08 16:22:50,477 - Database - INFO - Loaded biosphere3_spatialized_flows from brightway!


In [17]:
ranking_best_ecoinvent_locations_for_QC = [
    'CA-QC', # Quebec
    'CAN', # Canada in IMAGE and TIAM-UCL
    'CA', # Canada
    'NAM', # North America in MESSAGE
    'CAZ', # Canada - Australia - New Zealand in REMIND
    'RNA', # North America
    'US', # United States
    'USA', # United States in REMIND and IMAGE
    'CA-AB', # Alberta
    'GLO', # Global average 
    'RoW', # Rest of the world
]

In [18]:
# Add CPC categories to the main database
main_db.add_CPC_categories(mapping_new_products_to_CPC=mapping_new_product_to_cpc, overwrite_existing_CPC=True)

In [19]:
# Change the main database name in the mapping file
mapping['Database'] = name_main_database

In [20]:
esm = ESM(
    # Mandatory inputs
    mapping=mapping,
    unit_conversion=unit_conversion,
    model=model,
    mapping_esm_flows_to_CPC_cat=mapping_es_flows_to_cpc,
    main_database=main_db,
    esm_db_name=name_es_database,
    
    # Optional inputs
    technology_compositions=techno_compositions,
    tech_specifics=tech_specifics,
    lifetime=lifetime,
    efficiency=efficiency,
    regionalize_foregrounds=regionalize_foregrounds,
    accepted_locations=['CA-QC'],
    locations_ranking=ranking_best_ecoinvent_locations_for_QC,
    esm_location='CA-QC',
    results_path_file=LCA_RESULTS_DIR,
    biosphere_db_name=name_biosphere_db,
    
    # If we want regionalized results 
    spatialized_biosphere_db=spatialized_biosphere_db,
)

In [21]:
esm.clean_inputs()

In [22]:
# Adapt mapping file to ESM location
esm.change_location_mapping_file()
esm.mapping.to_csv('./Data/mapping_with_loc.csv', index=False)

In [23]:
missing_flows = main_db.test_mapping_file(esm.mapping)

2026-09-08 16:23:28,409 - Database - INFO - Mapping successfully linked to the database


In [24]:
missing_flows

[]

In [25]:
esm.check_inputs()

In [26]:
main_db = {}  # Free memory

### Generate ESM database

In [ ]:
# Foreground regionalization, double-counting removal, and efficiency harmonization
esm.create_esm_database()

## Generate LCA metrics

In [27]:
methods = [
    'IMPACT World+ Midpoint 2.2.1_regionalized for ecoinvent v3.12',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
]

### Life-cycle emissions

In [28]:
contrib_analysis = None  # 'emissions', 'processes', 'both' or None

In [29]:
# LCIA, Lifetime harmonization
if contrib_analysis is not None:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
        contribution_analysis=contrib_analysis,
        contribution_analysis_limit_type='number',
        contribution_analysis_limit=30,
    )
    contrib_analysis_emissions = contrib_analysis_res[contrib_analysis_res['contribution_type'] == 'emissions']
    contrib_analysis_processes = contrib_analysis_res[contrib_analysis_res['contribution_type'] == 'processes']
    contrib_analysis_emissions.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_emissions.csv', index=False)
    contrib_analysis_processes.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_processes.csv', index=False)
else:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
    )

Getting activity data


100%|██████████| 1740/1740 [00:00<00:00, 174074.87it/s]


Adding exchange data to activities


100%|██████████| 40881/40881 [00:01<00:00, 26222.46it/s]


Filling out exchange data


100%|██████████| 1740/1740 [00:01<00:00, 1440.88it/s]
2026-09-08 16:23:40,741 - Database - INFO - Loaded EnergyScope_CA-QC_image_SSP1-L_2050 from brightway!
1137it [03:50,  4.92it/s]


In [30]:
# Set the impact of transformers to zero for operation
R_long['Value'] = R_long.apply(lambda row: 0 if row['Name'].startswith('TRAFO_') and row['Type'] == 'Operation' else row['Value'], axis=1)

In [31]:
R_long.to_csv(f'{LCA_RESULTS_DIR}impact_scores.csv', index=False) # [impact / kW(h) or pkm(/h) or tkm(/h)]

### Direct emissions

In [32]:
R_long_direct_emissions, contrib_analysis_direct_emissions, _ = esm.compute_impact_scores(
    methods=methods,
    assessment_type='direct emissions',
    impact_abbrev=impact_abbrev,
    overwrite=True,
    contribution_analysis='emissions',
    contribution_analysis_limit=10,
)

Getting activity data


100%|██████████| 1740/1740 [00:00<00:00, 267045.59it/s]


Adding exchange data to activities


100%|██████████| 40881/40881 [00:01<00:00, 22703.51it/s]


Filling out exchange data


100%|██████████| 1740/1740 [00:01<00:00, 1511.08it/s]
2026-09-08 16:27:46,803 - Database - INFO - Loaded EnergyScope_CA-QC_image_SSP1-L_2050 from brightway!
Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 09/08/2026 16:28:22
  Finished: 09/08/2026 16:28:23
  Total time elapsed: 00:00:00
  CPU %: 98.20
  Memory %: 11.73


2026-09-08 16:29:39,181 - Database - INFO - EnergyScope_CA-QC_image_SSP1-L_2050_direct_emissions written to Brightway!
503it [01:35,  5.25it/s]


In [33]:
contrib_analysis_direct_emissions.to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_direct_emissions.csv', index=False)

In [34]:
# Set the impact of transformers to zero for operation
R_long_direct_emissions['Value'] = R_long_direct_emissions.apply(lambda row: 0 if row['Name'].startswith('TRAFO_') and row['Type'] == 'Operation' else row['Value'], axis=1)

In [35]:
R_long_direct_emissions.to_csv(f'{LCA_RESULTS_DIR}impact_scores_direct_emissions.csv', index=False)  # [impact / kW(h) or pkm(/h) or tkm(/h)]

### Territorial carbon emissions

In [36]:
if esm.esm_db is None:
    esm.esm_db = Database(esm.esm_db_name)

In [37]:
_, contrib_analysis_all_processes, _ = esm.compute_impact_scores(
    methods=methods,
    specific_lcia_abbrev=['m_CCS_all'],
    impact_abbrev=impact_abbrev,
    contribution_analysis='processes',
    contribution_analysis_limit_type='number',
    contribution_analysis_limit=2000,
)

Getting activity data


100%|██████████| 1740/1740 [00:00<00:00, 173751.61it/s]


Adding exchange data to activities


100%|██████████| 40881/40881 [00:01<00:00, 22791.44it/s]


Filling out exchange data


100%|██████████| 1740/1740 [00:01<00:00, 1314.52it/s]
2026-09-08 16:31:22,101 - Database - INFO - Loaded EnergyScope_CA-QC_image_SSP1-L_2050 from brightway!
1137it [18:43,  1.01it/s]


In [38]:
contrib_analysis_ccst = compute_territorial_emissions(contrib_analysis_all_processes, esm.main_database)

In [39]:
contrib_analysis_ccst.drop(columns=['amount']).to_csv(f'{LCA_RESULTS_DIR}contribution_analysis_all_processes_ccst.csv', index=False)

### Add a remaining AoP category (total AoP - climate change) and biogenic CC

In [42]:
R_long = pd.read_csv(f'{LCA_RESULTS_DIR}impact_scores.csv')
R_long_direct_emissions = pd.read_csv(f'{LCA_RESULTS_DIR}impact_scores_direct_emissions.csv')

In [43]:
R_long = add_biogenic_climate_change_to_impact_scores_df(R_long, ecoinvent_version = '3.12', iw_version = '2.2.1')
R_long = add_rhhd_and_reqd_to_impact_scores_df(R_long, ecoinvent_version = '3.12', iw_version = '2.2.1')
R_long_direct_emissions = add_biogenic_climate_change_to_impact_scores_df(R_long_direct_emissions, ecoinvent_version = '3.12', iw_version = '2.2.1')
R_long_direct_emissions = add_rhhd_and_reqd_to_impact_scores_df(R_long_direct_emissions, ecoinvent_version = '3.12', iw_version = '2.2.1')

In [44]:
R_long.to_csv(f'{LCA_RESULTS_DIR}impact_scores.csv', index=False)
R_long_direct_emissions.to_csv(f'{LCA_RESULTS_DIR}impact_scores_direct_emissions.csv', index=False)

## Create the .mod and .dat files

In [46]:
update_ampl_files(
    iam_scenarios = [{"model": iam, "pathway": ssp_rcp}],
    year = year,
    specific_lcia_abbrev = ['RHHD', 'REQD', 'm_CCS_all'],
    main_database = esm.main_database,
    direct_emissions_files = False,
    territorial_emissions_files = True,
    ecoinvent_version = ei_version,
    iw_version = '2.2.1',
)